In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

path = "/Volumes/projects/ecommerce_bronze/raw_files/online_retail_II.xlsx"

df_09_10 = pd.read_excel(path, sheet_name = "Year 2009-2010")
df_10_11 = pd.read_excel(path, sheet_name = "Year 2010-2011")

print(df_09_10.shape, df_10_11.shape)
df_10_11.head()

In [0]:
df_09_10["source_sheet"] = "2009-2010"
df_10_11["source_sheet"] = "2010-2011"
combined_pdf = pd.concat([df_09_10, df_10_11], ignore_index=True)

from datetime import datetime
combined_pdf['ingestion_timestamp'] = datetime.utcnow()
combined_pdf["_source_file"] = "online_retail_sales_II.xlsx"
print("combined_pdf shape : {combined_pdf.shape()}")
combined_pdf.dtypes

In [0]:
# Force StockCode, Invoice and Description to string - it has mixed str/int values in the raw data
combined_pdf["StockCode"] = combined_pdf["StockCode"].astype(str)
combined_pdf["Invoice"] = combined_pdf["Invoice"].astype(str)
combined_pdf["Description"] = combined_pdf["Description"].astype(str)
bronze_df = spark.createDataFrame(combined_pdf)
bronze_df.printSchema()
display(bronze_df.limit(10))

In [0]:
# Rename column to remove space (Delta doesn't allow spaces in column names by default)
bronze_df_cleaned = bronze_df.withColumnRenamed("Customer ID", "Customer_ID")

bronze_df_cleaned.write.format("delta")\
        .mode("overWrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable("projects.ecommerce_bronze.raw_retail_online")


In [0]:
print(f"Number of records in bronze table : {bronze_df_cleaned.count()}")
display(bronze_df_cleaned.groupBy("source_sheet").count())